# Data Cleaing For FRED Financial Market Volatility Project A
Forecast S&P 500 realized volatility using ARIMA, GARCH, Random Forest, with S&P returns, VIX, and gold volatility/GVZ as supporting variables.

Upload Data Files

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_PATH = Path(r"C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Market Volatility\Raw Data")

sp500 = pd.read_csv(RAW_DATA_PATH / "SP500.csv")
vix = pd.read_csv(RAW_DATA_PATH / "VIXCLS.csv")
gvz = pd.read_csv(RAW_DATA_PATH / "GVZCLS.csv")

print("SP500:")
display(sp500.head())
display(sp500.tail())

print("VIXCLS:")
display(vix.head())
display(vix.tail())

print("GVZCLS:")
display(gvz.head())
display(gvz.tail())

print("Shapes:")
print("SP500:", sp500.shape)
print("VIXCLS:", vix.shape)
print("GVZCLS:", gvz.shape)

SP500:


,observation_date,SP500
0,2016-05-02,2081.43
1,2016-05-03,2063.37
2,2016-05-04,2051.12
3,2016-05-05,2050.63
4,2016-05-06,2057.14


,observation_date,SP500
2604,2026-04-24,7165.08
2605,2026-04-27,7173.91
2606,2026-04-28,7138.80
2607,2026-04-29,7135.95
2608,2026-04-30,7209.01


VIXCLS:


,observation_date,VIXCLS
0,2016-04-29,15.70
1,2016-05-02,14.68
2,2016-05-03,15.60
3,2016-05-04,16.05
4,2016-05-05,15.91


,observation_date,VIXCLS
2604,2026-04-23,19.31
2605,2026-04-24,18.71
2606,2026-04-27,18.02
2607,2026-04-28,17.83
2608,2026-04-29,18.81


GVZCLS:


,observation_date,GVZCLS
0,2016-04-29,20.23
1,2016-05-02,19.29
2,2016-05-03,18.70
3,2016-05-04,19.47
4,2016-05-05,18.46


,observation_date,GVZCLS
2604,2026-04-23,27.87
2605,2026-04-24,25.88
2606,2026-04-27,25.37
2607,2026-04-28,26.19
2608,2026-04-29,27.77


Shapes:
SP500: (2609, 2)
VIXCLS: (2609, 2)
GVZCLS: (2609, 2)


## Data Cleaning and Preprocessing Plan

This analysis uses three daily financial time series downloaded from FRED: the S&P 500 Index (SP500), the CBOE Volatility Index (VIXCLS), and the CBOE Gold ETF Volatility Index (GVZCLS). Each dataset contains an observation date and a daily closing index value. The S&P 500 series is used to compute market returns and realized volatility, while VIX and GVZ are used as implied-volatility indicators for equity and gold-market risk.

The first cleaning step is to standardize the date column and convert it into a datetime object so that all series can be merged on a common trading-date index. The second step is to convert each value column into a numeric format, because financial datasets sometimes contain missing-value symbols or text-formatted numbers. The third step is to check for duplicated dates, missing observations, and date-range overlap across the three datasets.

The raw S&P 500 index level is not modeled directly. Instead, it is transformed into daily log returns, and then a 21-trading-day rolling realized volatility measure is computed. This transformation is necessary because raw stock-index levels are typically nonstationary, while time-series models such as ARMA/ARIMA and GARCH are designed around stationary or transformed series. The realized volatility measure becomes the primary target variable for the forecasting analysis.

VIX and GVZ are kept in their original index-level form because they are already volatility indexes. VIX represents market expectations of near-term equity volatility, while GVZ represents gold-market volatility. These variables are included as supporting predictors to test whether implied volatility from equity and gold markets helps forecast realized S&P 500 volatility.

Rows with missing values after merging and transformation are removed from the modeling dataset. This avoids using artificial forward-filled values and preserves a clean chronological structure for out-of-sample forecasting. The final cleaned dataset is sorted by date and used to create lagged predictors for ARIMA, GARCH, and Random Forest forecasting models.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_PATH = Path(r"C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Market Volatility\Raw Data")

def load_fred_csv(file_name, value_col_name):
    """
    Load one FRED CSV file, standardize date/value columns,
    convert values to numeric, sort by date, and check duplicates.
    """
    df = pd.read_csv(RAW_DATA_PATH / file_name)
    
    # Standardize column names
    df = df.rename(columns={
        "observation_date": "date",
        value_col_name: value_col_name.lower()
    })
    
    # Convert date column
    df["date"] = pd.to_datetime(df["date"])
    
    # Convert values to numeric; errors='coerce' turns bad values into NaN
    clean_value_col = value_col_name.lower()
    df[clean_value_col] = pd.to_numeric(df[clean_value_col], errors="coerce")
    
    # Sort by date
    df = df.sort_values("date").reset_index(drop=True)
    
    # Check for duplicate dates
    duplicate_count = df["date"].duplicated().sum()
    
    # Basic summary
    print(f"\n{value_col_name} summary")
    print("-" * 40)
    print("Rows:", len(df))
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("Missing values:", df[clean_value_col].isna().sum())
    print("Duplicate dates:", duplicate_count)
    print("Data type:", df[clean_value_col].dtype)
    
    display(df.head())
    display(df.tail())
    
    return df

sp500 = load_fred_csv("SP500.csv", "SP500")
vix = load_fred_csv("VIXCLS.csv", "VIXCLS")
gvz = load_fred_csv("GVZCLS.csv", "GVZCLS")


SP500 summary
----------------------------------------
Rows: 2609
Date range: 2016-05-02 00:00:00 to 2026-04-30 00:00:00
Missing values: 95
Duplicate dates: 0
Data type: float64


,date,sp500
0,2016-05-02,2081.43
1,2016-05-03,2063.37
2,2016-05-04,2051.12
3,2016-05-05,2050.63
4,2016-05-06,2057.14


,date,sp500
2604,2026-04-24,7165.08
2605,2026-04-27,7173.91
2606,2026-04-28,7138.80
2607,2026-04-29,7135.95
2608,2026-04-30,7209.01



VIXCLS summary
----------------------------------------
Rows: 2609
Date range: 2016-04-29 00:00:00 to 2026-04-29 00:00:00
Missing values: 66
Duplicate dates: 0
Data type: float64


,date,vixcls
0,2016-04-29,15.70
1,2016-05-02,14.68
2,2016-05-03,15.60
3,2016-05-04,16.05
4,2016-05-05,15.91


,date,vixcls
2604,2026-04-23,19.31
2605,2026-04-24,18.71
2606,2026-04-27,18.02
2607,2026-04-28,17.83
2608,2026-04-29,18.81



GVZCLS summary
----------------------------------------
Rows: 2609
Date range: 2016-04-29 00:00:00 to 2026-04-29 00:00:00
Missing values: 93
Duplicate dates: 0
Data type: float64


,date,gvzcls
0,2016-04-29,20.23
1,2016-05-02,19.29
2,2016-05-03,18.70
3,2016-05-04,19.47
4,2016-05-05,18.46


,date,gvzcls
2604,2026-04-23,27.87
2605,2026-04-24,25.88
2606,2026-04-27,25.37
2607,2026-04-28,26.19
2608,2026-04-29,27.77


In [8]:
# Inspect missing values before merging

def inspect_missing_values(df, value_col, series_name):
    """
    Print and display dates where a cleaned FRED series has missing values.
    """
    missing_rows = df[df[value_col].isna()].copy()
    
    print(f"\nMissing Value Inspection: {series_name}")
    print("-" * 60)
    print(f"Total missing values: {len(missing_rows)}")
    
    if len(missing_rows) == 0:
        print("No missing values found.")
        return missing_rows
    
    # Display missing dates vertically
    print("\nMissing dates shown vertically:")
    display(missing_rows[["date", value_col]].reset_index(drop=True))

    return missing_rows


sp500_missing = inspect_missing_values(sp500, "sp500", "SP500")
vix_missing = inspect_missing_values(vix, "vixcls", "VIXCLS")
gvz_missing = inspect_missing_values(gvz, "gvzcls", "GVZCLS")


Missing Value Inspection: SP500
------------------------------------------------------------
Total missing values: 95

Missing dates shown vertically:


,date,sp500
0,2016-05-30,NaN
1,2016-07-04,NaN
2,2016-09-05,NaN
3,2016-11-24,NaN
4,2016-12-26,NaN
...,...,...
90,2025-12-25,NaN
91,2026-01-01,NaN
92,2026-01-19,NaN
93,2026-02-16,NaN



Missing Value Inspection: VIXCLS
------------------------------------------------------------
Total missing values: 66

Missing dates shown vertically:


,date,vixcls
0,2016-05-30,NaN
1,2016-07-04,NaN
2,2016-09-05,NaN
3,2016-11-24,NaN
4,2016-12-26,NaN
...,...,...
61,2025-01-01,NaN
62,2025-04-18,NaN
63,2025-12-25,NaN
64,2026-01-01,NaN



Missing Value Inspection: GVZCLS
------------------------------------------------------------
Total missing values: 93

Missing dates shown vertically:


,date,gvzcls
0,2016-05-30,NaN
1,2016-07-04,NaN
2,2016-09-05,NaN
3,2016-11-24,NaN
4,2016-12-26,NaN
...,...,...
88,2025-12-25,NaN
89,2026-01-01,NaN
90,2026-01-19,NaN
91,2026-02-16,NaN


In [5]:
# Compare whether missing dates overlap across the three series

sp500_missing_dates = set(sp500_missing["date"])
vix_missing_dates = set(vix_missing["date"])
gvz_missing_dates = set(gvz_missing["date"])

common_missing_all = sp500_missing_dates & vix_missing_dates & gvz_missing_dates

print("Missing-date overlap summary")
print("-" * 60)
print("SP500 missing:", len(sp500_missing_dates))
print("VIXCLS missing:", len(vix_missing_dates))
print("GVZCLS missing:", len(gvz_missing_dates))
print("Missing in all three:", len(common_missing_all))

if len(common_missing_all) > 0:
    common_missing_all_df = pd.DataFrame(
        sorted([d.strftime("%Y-%m-%d") for d in common_missing_all]),
        columns=["date_missing_in_all_three"]
    )
    display(common_missing_all_df)

Missing-date overlap summary
------------------------------------------------------------
SP500 missing: 95
VIXCLS missing: 66
GVZCLS missing: 93
Missing in all three: 64


,date_missing_in_all_three
0,2016-05-30
1,2016-07-04
2,2016-09-05
3,2016-11-24
4,2016-12-26
...,...
59,2025-01-01
60,2025-04-18
61,2025-12-25
62,2026-01-01


Missing values were identified after converting FRED series values to numeric format. Rather than manually editing raw CSV files, missing observations were handled programmatically after merging the series on date. This preserved reproducibility and ensured the modeling dataset used only dates with complete information across the selected variables.

In [9]:
# Inspect missing values by weekday

def missing_weekday_summary(missing_df, series_name):
    """
    Summarize missing-value dates by day of week.
    This helps distinguish weekends/holidays from irregular missing data.
    """
    if missing_df.empty:
        print(f"\n{series_name}: No missing values.")
        return
    
    temp = missing_df.copy()
    temp["weekday"] = temp["date"].dt.day_name()
    temp["year"] = temp["date"].dt.year
    
    print(f"\n{series_name} missing values by weekday")
    print("-" * 60)
    display(temp["weekday"].value_counts().rename_axis("weekday").reset_index(name="missing_count"))
    
    print(f"\n{series_name} missing values by year")
    print("-" * 60)
    display(temp["year"].value_counts().sort_index().rename_axis("year").reset_index(name="missing_count"))

missing_weekday_summary(sp500_missing, "SP500")
missing_weekday_summary(vix_missing, "VIXCLS")
missing_weekday_summary(gvz_missing, "GVZCLS")


SP500 missing values by weekday
------------------------------------------------------------


,weekday,missing_count
0,Monday,53
1,Thursday,16
2,Friday,15
3,Wednesday,7
4,Tuesday,4



SP500 missing values by year
------------------------------------------------------------


,year,missing_count
0,2016,5
1,2017,9
2,2018,10
3,2019,9
4,2020,9
5,2021,9
6,2022,9
7,2023,10
8,2024,10
9,2025,11



VIXCLS missing values by weekday
------------------------------------------------------------


...

,year,missing_count
0,2016,5
1,2017,9
2,2018,10
3,2019,9
4,2020,9
5,2021,9
6,2022,4
7,2023,3
8,2024,3
9,2025,3



GVZCLS missing values by weekday
------------------------------------------------------------


,weekday,missing_count
0,Monday,53
1,Thursday,16
2,Friday,13
3,Wednesday,7
4,Tuesday,4



GVZCLS missing values by year
------------------------------------------------------------


,year,missing_count
0,2016,5
1,2017,9
2,2018,10
3,2019,9
4,2020,9
5,2021,7
6,2022,9
7,2023,10
8,2024,10
9,2025,11


In [10]:
# ------------------------------------------------------------
# Identify missing values unique to each series
# ------------------------------------------------------------

sp500_set = set(sp500_missing["date"])
vix_set = set(vix_missing["date"])
gvz_set = set(gvz_missing["date"])

missing_sets = {
    "SP500_only": sp500_set - vix_set - gvz_set,
    "VIXCLS_only": vix_set - sp500_set - gvz_set,
    "GVZCLS_only": gvz_set - sp500_set - vix_set,
    "SP500_and_VIXCLS_only": (sp500_set & vix_set) - gvz_set,
    "SP500_and_GVZCLS_only": (sp500_set & gvz_set) - vix_set,
    "VIXCLS_and_GVZCLS_only": (vix_set & gvz_set) - sp500_set,
    "All_three": sp500_set & vix_set & gvz_set
}

summary = pd.DataFrame({
    "category": list(missing_sets.keys()),
    "count": [len(v) for v in missing_sets.values()]
})

display(summary)

for category, dates in missing_sets.items():
    if len(dates) > 0:
        print(f"\n{category}: {len(dates)} dates")
        temp = pd.DataFrame(sorted([d.strftime("%Y-%m-%d") for d in dates]), columns=["date"])
        display(temp)

,category,count
0,SP500_only,0
1,VIXCLS_only,0
2,GVZCLS_only,0
3,SP500_and_VIXCLS_only,2
4,SP500_and_GVZCLS_only,29
5,VIXCLS_and_GVZCLS_only,0
6,All_three,64



SP500_and_VIXCLS_only: 2 dates


,date
0,2021-04-02
1,2021-12-24



SP500_and_GVZCLS_only: 29 dates


,date
0,2022-05-30
1,2022-06-20
2,2022-07-04
3,2022-09-05
4,2022-11-24
5,2023-01-16
6,2023-02-20
7,2023-05-29
8,2023-06-19
9,2023-07-04



All_three: 64 dates


,date
0,2016-05-30
1,2016-07-04
2,2016-09-05
3,2016-11-24
4,2016-12-26
...,...
59,2025-01-01
60,2025-04-18
61,2025-12-25
62,2026-01-01


### Missing Value Inspection

Missing values were inspected before merging the three FRED series: S&P 500, VIX, and GVZ. The purpose of this step was to distinguish ordinary non-reporting dates, such as holidays or market closures, from irregular data gaps. Because these are daily financial market series, missing observations are expected around non-trading days.

The raw CSV files were not manually edited. Instead, each value column was converted to numeric format, which allowed invalid or unavailable observations to be identified programmatically as missing values. Missing observations were then summarized by series, weekday, year, and overlap across the S&P 500, VIX, and GVZ datasets.

The missing-value inspection showed that most missing observations occurred across multiple series on the same dates. This pattern suggests that the gaps were primarily calendar-related rather than isolated data quality problems. Since the forecasting models require complete observations across all selected variables, incomplete rows will be removed after merging the datasets by date. This preserves a reproducible cleaning process and avoids subjective manual changes to the raw files.

The missing-value diagnostics support using a complete-case approach after merging. Since most missing values appear to be related to shared non-trading dates or market-calendar differences, the analysis will drop rows with missing values after the datasets are merged on date. This ensures that ARIMA, GARCH, and Random Forest models are trained only on dates with complete S&P 500, VIX, and GVZ information.

## Merge Cleaned Series and Construct Volatility Target

# Merge cleaned FRED series and construct S&P 500 volatility target

from pathlib import Path
import numpy as np
import pandas as pd

# Folder where cleaned modeling dataset will be saved
CLEAN_DATA_PATH = Path(r"C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Market Volatility\Clean Data")
CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Merge the three cleaned datasets by date
# Inner join keeps only dates that appear in all three datasets
market_data = (
    sp500
    .merge(vix, on="date", how="inner")
    .merge(gvz, on="date", how="inner")
)

print("Merged dataset before dropping missing values:")
print("Rows:", len(market_data))
print("Columns:", market_data.columns.tolist())
print("Missing values by column:")
display(market_data.isna().sum())

# Drop rows with missing values across required variables
market_data_clean = market_data.dropna(subset=["sp500", "vixcls", "gvzcls"]).copy()

print("\nMerged dataset after dropping missing values:")
print("Rows:", len(market_data_clean))
print("Rows removed:", len(market_data) - len(market_data_clean))
print("Date range:", market_data_clean["date"].min(), "to", market_data_clean["date"].max())
display(market_data_clean.head())
display(market_data_clean.tail())

# Compute S&P 500 daily log returns
market_data_clean["sp500_log_return"] = np.log(
    market_data_clean["sp500"] / market_data_clean["sp500"].shift(1)
)

# Compute 21-trading-day realized volatility and annualize it
# 21 trading days approximates one trading month
# sqrt(252) annualizes daily volatility using approximately 252 trading days per year
market_data_clean["sp500_realized_vol_21d"] = (
    market_data_clean["sp500_log_return"].rolling(window=21).std() * np.sqrt(252)
)

# Remove rows created as missing by return and rolling-volatility calculations
model_data = market_data_clean.dropna().copy()

print("\nFinal cleaned modeling dataset:")
print("Rows:", len(model_data))
print("Columns:", model_data.columns.tolist())
print("Date range:", model_data["date"].min(), "to", model_data["date"].max())
display(model_data.head())
display(model_data.tail())

# Save the cleaned modeling dataset
output_file = CLEAN_DATA_PATH / "analysis_a_market_volatility_clean.csv"
model_data.to_csv(output_file, index=False)

print(f"\nCleaned modeling dataset saved to:\n{output_file}")

## Merging and Volatility Target Construction

The final cleaned dataset for Analysis A contains six variables: `date`, `sp500`, `vixcls`, `gvzcls`, `sp500_log_return`, and `sp500_realized_vol_21d`. The raw FRED files were not manually edited. Instead, each file was loaded into Python, the date column was converted to datetime format, value columns were converted to numeric format, duplicate dates were checked, missing values were inspected, and the three series were merged by the common `date` column.

An inner join was used so that only dates appearing across the selected series were retained. Rows with incomplete observations were removed after merging. This complete-case approach was used because the forecasting models require S&P 500, VIX, and GVZ information to be available on the same trading date.

The S&P 500 index level is retained as the base equity-market series. FRED identifies the S&P 500 series as a daily close index value sourced from S&P Dow Jones Indices LLC. FRED also notes that the S&P 500 observations represent the daily index value at market close, making the series appropriate for constructing daily financial returns. 

The raw S&P 500 level was not used directly as the forecasting target. Stock-index levels often contain trends, structural breaks, and nonstationary behavior. This is consistent with the course framework, where stationarity and temporal dependence are central concerns for ARMA/ARIMA-style modeling. Therefore, the S&P 500 level was transformed into daily log returns:

$$
r_t = \log\left(\frac{P_t}{P_{t-1}}\right)
$$

where `r_t` is the S&P 500 log return on day `t`, `P_t` is the S&P 500 closing value on day `t`, and `P_{t-1}` is the previous trading day’s closing value. In plain English, this measures the daily percentage-like change in the S&P 500. Log returns are used because they convert the raw index level into a return series that is more appropriate for financial time-series modeling.

The main forecasting target is 21-day realized volatility:

$$
RV_{t,21} = \sqrt{252} \times SD(r_{t-20}, \ldots, r_t)
$$

where `SD` is the standard deviation of S&P 500 log returns over the most recent 21 trading days. The 21-day window approximates one trading month, and multiplying by the square root of 252 annualizes the volatility estimate using the common assumption of approximately 252 trading days per year. Realized volatility is appropriate here because the project objective is to forecast market volatility rather than raw stock prices. Andersen and Benzoni describe realized volatility as a nonparametric ex-post estimate of return variation, which supports using return-based volatility as the observed target for volatility forecasting.

VIX and GVZ are retained as supporting volatility indicators. FRED identifies VIX as the CBOE Volatility Index and states that it measures the market expectation of near-term volatility conveyed by stock-index option prices. GVZ is included as the CBOE Gold ETF Volatility Index, giving the model a cross-asset volatility measure related to gold-market uncertainty. These indicators allow the analysis to test whether implied equity volatility and gold-market volatility improve forecasts of realized S&P 500 volatility.

### Supporting References for Data Sources and Transformations
Andersen, T. G., & Benzoni, L. (2008). Realized volatility. Federal Reserve Bank of Chicago Working Paper No. 2008-14.

Chicago Board Options Exchange. (n.d.). CBOE Gold ETF Volatility Index [GVZCLS]. Federal Reserve Bank of St. Louis, FRED.

Chicago Board Options Exchange. (n.d.). CBOE Volatility Index: VIX [VIXCLS]. Federal Reserve Bank of St. Louis, FRED.

S&P Dow Jones Indices LLC. (n.d.). S&P 500 [SP500]. Federal Reserve Bank of St. Louis, FRED.